# release16_checkpoint_distillation_bc_readiness_v1

Purpose: test whether the `release16_checkpoint_distillation_ytd_alignment_contract_v1` target contract is immediately usable as a BC warm-start/training handoff before spending a full PPO run.

Governance:
- Canonical baseline: `soxlbacktest_policy_reimplementation_v1`.
- Direct comparison: `release16_checkpoint_distillation_ytd_alignment_contract_v1` and the latest retention gate summary.
- Role: `diagnose`.
- Target windows: `wf_2020`, `wf_2022`, `wf_2023_2024` train-local target folds.
- Changed layer: `model_selection` / `training_objective_contract` / `bc_warm_start_readiness`.
- External reference: converted qlib SOXLBacktest buy/sell baseline, release16 state-objective target contract, and existing `state_objective_bc` helper.

Hypothesis: if the fold-local target contract can be fit by the existing state-objective BC actor with acceptable release/downtrend/rebound slice accuracy and no label-scope leakage, the next useful step is wiring this contract into a retained-actor training run with PPO drift stops. If not, target construction or weighting must be repaired before another PPO loop.

Acceptance criteria for this readiness screen:
- Target scope audit passes: training labels only, no eval/YTD direct labels.
- Each walk-forward fold has non-empty train-local labels and feature-frame coverage.
- BC training improves loss and reaches minimum overall/slice accuracy thresholds.
- No actor is retained or promoted from this notebook.


In [1]:

from __future__ import annotations

import json
import math
import random
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "SPEC.md").exists() and (candidate / "src/soxl_quant").exists():
            return candidate
    raise RuntimeError(f"could not locate repo root from {start}")

ROOT = find_repo_root(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCSampleWeightConfig,
    StateObjectiveBCTrainingConfig,
    build_state_objective_bc_dataset_from_frame,
    evaluate_bc_actor_slice_fit,
    fit_state_objective_bc_actor_from_frame,
    state_objective_bc_sample_weights,
)
from src.soxl_quant.config import load_profile
from src.soxl_quant.data import build_dataset, feature_columns_for_packs, init_qlib, segment_frame
from src.soxl_quant.policy.positioning import action_levels_for_profile
from src.soxl_quant.settings import QLIB_DATA_PATH

EXPERIMENT = "release16_checkpoint_distillation_bc_readiness_v1"
CARD = ROOT / "runtime/model_selection/card" / EXPERIMENT
SOURCE_CARD = ROOT / "runtime/model_selection/card/release16_checkpoint_distillation_ytd_alignment_contract_v1"
RETENTION_GATE = ROOT / "runtime/model_selection/card/model_retention_walkforward_gate_v1/summary.json"
TARGET_CONTRACT = SOURCE_CARD / "distillation_target_contract.csv"
PROFILE_NAME = "sparse_release_bc_ppo_screen_v1"
SEED = 20260615

CARD.mkdir(parents=True, exist_ok=True)

BC_TRAIN_CONFIG = StateObjectiveBCTrainingConfig(steps=400, lr=0.001, batch_size=128, seed=SEED)
BC_WEIGHT_CONFIG = StateObjectiveBCSampleWeightConfig(
    rebound_multiplier=6.0,
    downtrend_multiplier=2.0,
    release_gate_multiplier=16.0,
    objective_weight_power=0.20,
    class_balanced_beta=0.99,
    weight_cap=35.0,
)

WALK_FORWARD_WINDOWS = {
    "wf_2020": {
        "target_fold": "wf_2020_train",
        "train": ("2010-03-11", "2018-12-31"),
        "valid": ("2019-01-01", "2019-12-31"),
        "test": ("2020-01-01", "2020-12-31"),
    },
    "wf_2022": {
        "target_fold": "wf_2022_train",
        "train": ("2010-03-11", "2020-12-31"),
        "valid": ("2021-01-01", "2021-12-31"),
        "test": ("2022-01-01", "2022-12-31"),
    },
    "wf_2023_2024": {
        "target_fold": "wf_2023_2024_train",
        "train": ("2010-03-11", "2022-12-31"),
        "valid": ("2023-01-01", "2023-12-31"),
        "test": ("2024-01-01", "2024-12-31"),
    },
}

SLICE_RULES = {
    "all": {"min_samples": 100, "min_accuracy": 0.55, "required": True},
    "release_gate_state": {"min_samples": 5, "min_accuracy": 0.60, "required": True},
    "is_rebound_cluster": {"min_samples": 5, "min_accuracy": 0.55, "required": False},
    "is_downtrend_state": {"min_samples": 100, "min_accuracy": 0.65, "required": False},
    "drawdown_control_state": {"min_samples": 20, "min_accuracy": 0.55, "required": False},
}
SLICE_COLS = tuple(k for k in SLICE_RULES if k != "all")


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + "\n", encoding="utf-8")


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def normalize_bool_series(series: pd.Series) -> pd.Series:
    if series.dtype == object:
        return series.fillna(False).astype(str).str.lower().isin({"true", "1", "yes", "y", "t"})
    return series.fillna(False).astype(bool)


def frame_date_bounds(frame: pd.DataFrame) -> tuple[str | None, str | None]:
    idx = frame.index
    if isinstance(idx, pd.MultiIndex):
        level = "datetime" if "datetime" in idx.names else 0
        values = idx.get_level_values(level)
    else:
        values = idx
    dates = pd.to_datetime(values).normalize()
    if len(dates) == 0:
        return None, None
    return str(dates.min().date()), str(dates.max().date())


def build_replay_feature_frame(profile):
    dataset = build_dataset(
        train_seg=("2010-03-11", "2022-12-31"),
        valid_seg=("2023-01-01", "2023-12-31"),
        test_seg=("2024-01-01", "2024-12-31"),
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    parts = [segment_frame(dataset, name) for name in ("train", "valid", "test")]
    frame = pd.concat(parts).sort_index()
    return frame.loc[~frame.index.duplicated(keep="last")].copy()


def slice_gate_from_metrics(slice_metrics: pd.DataFrame) -> tuple[bool, dict[str, Any]]:
    details: dict[str, Any] = {}
    failed: list[str] = []
    indexed = slice_metrics.set_index("slice") if not slice_metrics.empty else pd.DataFrame()
    for name, rule in SLICE_RULES.items():
        if name not in indexed.index:
            row = {"samples": 0, "accuracy": None, "loss": None, "gate": False, **rule}
        else:
            metric = indexed.loc[name]
            samples = int(metric.get("samples", 0))
            accuracy = float(metric.get("accuracy", float("nan")))
            loss = float(metric.get("loss", float("nan")))
            enough_samples = samples >= int(rule["min_samples"])
            enough_accuracy = math.isfinite(accuracy) and accuracy >= float(rule["min_accuracy"])
            row = {
                "samples": samples,
                "accuracy": accuracy,
                "loss": loss,
                "gate": bool(enough_samples and enough_accuracy),
                **rule,
            }
        details[name] = row
        if bool(rule["required"]) and not bool(row["gate"]):
            failed.append(name)
    return len(failed) == 0, {"failed_required_slices": failed, "slice_details": details}

print("root", ROOT)
print("card", CARD)


root /Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix
card /Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/runtime/model_selection/card/release16_checkpoint_distillation_bc_readiness_v1


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:

profile = load_profile(PROFILE_NAME)
targets = pd.read_csv(TARGET_CONTRACT)
targets["datetime"] = pd.to_datetime(targets["datetime"]).dt.normalize()
for col in ("training_label_allowed", "is_rebound_cluster", "is_downtrend_state", "release_gate_state", "drawdown_control_state"):
    if col in targets.columns:
        targets[col] = normalize_bool_series(targets[col])

source_selection = load_json(SOURCE_CARD / "selection.json")
retention_gate = load_json(RETENTION_GATE)
feature_columns = feature_columns_for_packs(profile.feature.packs, profile.name)
action_levels = list(action_levels_for_profile(profile))

scope_audit = {
    "rows": int(len(targets)),
    "training_label_allowed_rows": int(targets.get("training_label_allowed", pd.Series(False, index=targets.index)).sum()),
    "blocked_training_label_rows": int((~targets.get("training_label_allowed", pd.Series(False, index=targets.index))).sum()),
    "target_scope_values": sorted(targets.get("target_scope", pd.Series(dtype=object)).dropna().astype(str).unique().tolist()),
    "label_role_values": sorted(targets.get("label_role", pd.Series(dtype=object)).dropna().astype(str).unique().tolist()),
    "oos_guard_values": sorted(targets.get("oos_or_ytd_direct_label_guard", pd.Series(dtype=object)).dropna().astype(str).unique().tolist()),
    "min_date": str(targets["datetime"].min().date()),
    "max_date": str(targets["datetime"].max().date()),
}
scope_audit["target_scope_gate"] = (
    scope_audit["blocked_training_label_rows"] == 0
    and scope_audit["target_scope_values"] == ["fold_local"]
    and scope_audit["oos_guard_values"] == ["allowed_train_dates_only"]
)

pd.DataFrame([scope_audit]).to_csv(CARD / "target_scope_audit.csv", index=False)
print("profile", profile.name, "feature_packs", tuple(profile.feature.packs), "feature_count", len(feature_columns), "action_levels", action_levels)
print("scope_audit", scope_audit)


profile sparse_release_bc_ppo_screen_v1 feature_packs () feature_count 59 action_levels [np.float32(0.0), np.float32(0.1), np.float32(0.2), np.float32(0.3), np.float32(0.4), np.float32(0.5), np.float32(0.6), np.float32(0.7), np.float32(0.8), np.float32(0.9), np.float32(1.0)]
scope_audit {'rows': 3731, 'training_label_allowed_rows': 3731, 'blocked_training_label_rows': 0, 'target_scope_values': ['fold_local'], 'label_role_values': ['fold_local_train_label'], 'oos_guard_values': ['allowed_train_dates_only'], 'min_date': '2010-03-11', 'max_date': '2022-12-30', 'target_scope_gate': True}


In [3]:

init_qlib(QLIB_DATA_PATH)
frame = build_replay_feature_frame(profile)
frame_min, frame_max = frame_date_bounds(frame)
feature_schema = pd.DataFrame({"feature": feature_columns})
feature_schema.to_csv(CARD / "feature_schema.csv", index=False)
feature_summary = {
    "profile": profile.name,
    "feature_packs": list(profile.feature.packs),
    "feature_count": int(len(feature_columns)),
    "feature_frame_rows": int(len(frame)),
    "feature_frame_columns": int(len(frame.columns)),
    "feature_frame_min_date": frame_min,
    "feature_frame_max_date": frame_max,
}
write_json(CARD / "feature_summary.json", feature_summary)
print(feature_summary)


[22069:MainThread](2026-06-15 03:41:56,913) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[22069:MainThread](2026-06-15 03:41:57,314) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[22069:MainThread](2026-06-15 03:41:57,315) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/data/qlib/soxl')}


[22069:MainThread](2026-06-15 03:41:58,531) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[22069:MainThread](2026-06-15 03:41:58,531) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[22069:MainThread](2026-06-15 03:41:58,531) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | Init data Done


{'profile': 'sparse_release_bc_ppo_screen_v1', 'feature_packs': [], 'feature_count': 59, 'feature_frame_rows': 3475, 'feature_frame_columns': 59, 'feature_frame_min_date': '2011-03-11', 'feature_frame_max_date': '2024-12-31'}


In [4]:

fold_rows: list[dict[str, Any]] = []
slice_frames: list[pd.DataFrame] = []
action_frames: list[pd.DataFrame] = []
dataset_rows: list[dict[str, Any]] = []
weight_rows: list[dict[str, Any]] = []

for fold, spec in WALK_FORWARD_WINDOWS.items():
    print("fold", fold)
    set_seed(SEED)
    train_start = pd.Timestamp(spec["train"][0])
    train_end = pd.Timestamp(spec["train"][1])
    valid_start = pd.Timestamp(spec["valid"][0])
    valid_end = pd.Timestamp(spec["valid"][1])
    test_start = pd.Timestamp(spec["test"][0])
    test_end = pd.Timestamp(spec["test"][1])

    fold_all = targets[targets["fold"].astype(str).eq(spec["target_fold"])].copy()
    fold_targets = fold_all[fold_all["datetime"].between(train_start, train_end)].copy()
    valid_overlap = fold_all[fold_all["datetime"].between(valid_start, valid_end)]
    test_overlap = fold_all[fold_all["datetime"].between(test_start, test_end)]
    label_allowed = fold_targets.get("training_label_allowed", pd.Series(False, index=fold_targets.index)).fillna(False).astype(bool)
    fold_targets = fold_targets[label_allowed].copy()

    target_min = str(fold_targets["datetime"].min().date()) if not fold_targets.empty else None
    target_max = str(fold_targets["datetime"].max().date()) if not fold_targets.empty else None
    dataset_row = {
        "fold": fold,
        "target_fold": spec["target_fold"],
        "target_rows": int(len(fold_targets)),
        "valid_overlap_rows": int(len(valid_overlap)),
        "test_overlap_rows": int(len(test_overlap)),
        "target_min_date": target_min,
        "target_max_date": target_max,
        "target_scope_ok": bool(len(fold_targets) > 0 and len(test_overlap) == 0),
    }

    if fold_targets.empty:
        dataset_rows.append(dataset_row)
        fold_rows.append({"fold": fold, "status": "blocked_no_fold_targets", **dataset_row})
        continue

    bc_dataset = build_state_objective_bc_dataset_from_frame(
        frame,
        targets=fold_targets,
        profile=profile,
        sample_size=0,
        rng_seed=SEED,
    )
    weights = state_objective_bc_sample_weights(bc_dataset.metadata, actions=bc_dataset.actions, config=BC_WEIGHT_CONFIG)
    dataset_row.update(
        {
            "bc_samples": int(len(bc_dataset.actions)),
            "metadata_rows": int(len(bc_dataset.metadata)),
            "unique_label_actions": int(pd.Series(bc_dataset.actions).nunique()),
            "feature_coverage_ok": bool(len(bc_dataset.actions) > 0),
        }
    )
    dataset_rows.append(dataset_row)
    weight_rows.append(
        {
            "fold": fold,
            "weight_mean": float(weights.mean()) if len(weights) else None,
            "weight_min": float(weights.min()) if len(weights) else None,
            "weight_max": float(weights.max()) if len(weights) else None,
            "weight_p95": float(np.quantile(weights, 0.95)) if len(weights) else None,
        }
    )

    bc_result = fit_state_objective_bc_actor_from_frame(
        frame,
        targets=fold_targets,
        profile=profile,
        train_config=BC_TRAIN_CONFIG,
        sample_weight_config=BC_WEIGHT_CONFIG,
        sample_size=0,
        rng_seed=SEED,
    )
    slice_metrics = evaluate_bc_actor_slice_fit(
        bc_result.actor,
        bc_result.obs,
        bc_result.actions,
        bc_dataset.metadata,
        slice_cols=SLICE_COLS,
    )
    slice_gate, slice_details = slice_gate_from_metrics(slice_metrics)
    tagged_slices = slice_metrics.copy()
    tagged_slices.insert(0, "fold", fold)
    slice_frames.append(tagged_slices)

    action_distribution = bc_result.action_distribution.copy()
    action_distribution.insert(0, "fold", fold)
    if action_levels:
        action_distribution["action_level"] = action_distribution["action"].map(
            {idx: level for idx, level in enumerate(action_levels)}
        )
    action_frames.append(action_distribution)

    before_loss = float(bc_result.metrics["before_loss"])
    after_loss = float(bc_result.metrics["after_loss"])
    before_accuracy = float(bc_result.metrics["before_accuracy"])
    after_accuracy = float(bc_result.metrics["after_accuracy"])
    fit_gate = bool(after_loss < before_loss and after_accuracy >= SLICE_RULES["all"]["min_accuracy"] and slice_gate)
    fold_rows.append(
        {
            "fold": fold,
            "status": "evaluated",
            "fit_gate": fit_gate,
            "slice_gate": bool(slice_gate),
            "failed_required_slices": ",".join(slice_details["failed_required_slices"]),
            "before_loss": before_loss,
            "after_loss": after_loss,
            "loss_improvement": before_loss - after_loss,
            "before_accuracy": before_accuracy,
            "after_accuracy": after_accuracy,
            "train_last_loss": float(bc_result.metrics["train_last_loss"]),
            "train_last_batch_accuracy": float(bc_result.metrics["train_last_batch_accuracy"]),
            "samples": int(bc_result.metrics["samples"]),
            "steps": int(bc_result.metrics["steps"]),
            "lr": float(bc_result.metrics["lr"]),
            "batch_size": int(bc_result.metrics["batch_size"]),
            "weight_mean": float(bc_result.metrics.get("weight_mean", float("nan"))),
            "weight_min": float(bc_result.metrics.get("weight_min", float("nan"))),
            "weight_max": float(bc_result.metrics.get("weight_max", float("nan"))),
            **dataset_row,
        }
    )

fold_metrics = pd.DataFrame(fold_rows)
dataset_summary = pd.DataFrame(dataset_rows)
weight_summary = pd.DataFrame(weight_rows)
slice_metrics_all = pd.concat(slice_frames, ignore_index=True) if slice_frames else pd.DataFrame()
action_distribution_all = pd.concat(action_frames, ignore_index=True) if action_frames else pd.DataFrame()

fold_metrics.to_csv(CARD / "fold_bc_metrics.csv", index=False)
dataset_summary.to_csv(CARD / "dataset_summary.csv", index=False)
weight_summary.to_csv(CARD / "sample_weight_summary.csv", index=False)
slice_metrics_all.to_csv(CARD / "state_slice_bc_metrics.csv", index=False)
action_distribution_all.to_csv(CARD / "bc_action_distribution.csv", index=False)

display(dataset_summary)
display(fold_metrics)
display(slice_metrics_all)
display(action_distribution_all)


fold wf_2020


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


fold wf_2022


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


fold wf_2023_2024


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


,fold,target_fold,target_rows,valid_overlap_rows,test_overlap_rows,target_min_date,target_max_date,target_scope_ok,bc_samples,metadata_rows,unique_label_actions,feature_coverage_ok
0,wf_2020,wf_2020_train,2218,252,0,2010-03-11,2018-12-31,True,1905,1905,5,True
1,wf_2022,wf_2022_train,253,252,0,2020-01-02,2020-12-31,True,253,253,5,True
2,wf_2023_2024,wf_2023_2024_train,756,0,0,2020-01-02,2022-12-30,True,756,756,5,True


,fold,status,fit_gate,slice_gate,failed_required_slices,before_loss,after_loss,loss_improvement,before_accuracy,after_accuracy,...,target_rows,valid_overlap_rows,test_overlap_rows,target_min_date,target_max_date,target_scope_ok,bc_samples,metadata_rows,unique_label_actions,feature_coverage_ok
0,wf_2020,evaluated,True,True,,1.605351,1.183302,0.422050,0.304987,0.724934,...,2218,252,0,2010-03-11,2018-12-31,True,1905,1905,5,True
1,wf_2022,evaluated,True,True,,1.614201,1.032363,0.581837,0.138340,0.873518,...,253,252,0,2020-01-02,2020-12-31,True,253,253,5,True
2,wf_2023_2024,evaluated,False,False,release_gate_state,1.613803,1.167221,0.446582,0.074074,0.742063,...,756,0,0,2020-01-02,2022-12-30,True,756,756,5,True


,fold,slice,samples,accuracy,loss,label_unique_actions,predicted_unique_actions,label_dominant_action,label_dominant_share,predicted_dominant_action,predicted_dominant_share
0,wf_2020,all,1905,0.724934,1.183302,5,3,4,0.338058,4,0.440945
1,wf_2020,drawdown_control_state,708,0.844633,1.061420,4,3,2,0.524011,2,0.556497
2,wf_2020,is_downtrend_state,708,0.844633,1.061420,4,3,2,0.524011,2,0.556497
3,wf_2020,is_rebound_cluster,32,0.750000,1.147921,3,3,4,0.843750,4,0.718750
4,wf_2020,release_gate_state,32,0.750000,1.147921,3,3,4,0.843750,4,0.718750
5,wf_2022,all,253,0.873518,1.032363,5,5,0,0.418972,0,0.466403
6,wf_2022,drawdown_control_state,0,NaN,NaN,0,0,-1,1.000000,-1,1.000000
7,wf_2022,is_downtrend_state,118,0.949153,0.955480,5,5,1,0.457627,1,0.423729
8,wf_2022,is_rebound_cluster,12,0.916667,0.983795,5,4,4,0.500000,4,0.500000
9,wf_2022,release_gate_state,8,0.875000,1.023031,4,3,4,0.375000,4,0.375000


,fold,action,count,share,action_level
0,wf_2020,0,496,0.260367,0.0
1,wf_2020,1,68,0.035696,0.1
2,wf_2020,2,581,0.304987,0.2
3,wf_2020,3,116,0.060892,0.3
4,wf_2020,4,644,0.338058,0.4
5,wf_2022,0,106,0.418972,0.0
6,wf_2022,1,79,0.312253,0.1
7,wf_2022,2,35,0.138340,0.2
8,wf_2022,3,5,0.019763,0.3
9,wf_2022,4,28,0.110672,0.4


In [5]:

required_folds = set(WALK_FORWARD_WINDOWS)
evaluated = fold_metrics[fold_metrics["status"].eq("evaluated")].copy() if not fold_metrics.empty else pd.DataFrame()
all_required_folds_evaluated = set(evaluated.get("fold", pd.Series(dtype=str)).astype(str)) == required_folds
all_fit_gates = bool(all_required_folds_evaluated and evaluated["fit_gate"].fillna(False).astype(bool).all())
all_target_scope_ok = bool(scope_audit["target_scope_gate"] and dataset_summary["target_scope_ok"].fillna(False).astype(bool).all())
all_feature_coverage_ok = bool(dataset_summary.get("feature_coverage_ok", pd.Series(False, index=dataset_summary.index)).fillna(False).astype(bool).all())
release_gate_metrics = slice_metrics_all[slice_metrics_all["slice"].eq("release_gate_state")].copy() if not slice_metrics_all.empty else pd.DataFrame()
release_gate_pass_count = int((release_gate_metrics["accuracy"] >= SLICE_RULES["release_gate_state"]["min_accuracy"]).sum()) if not release_gate_metrics.empty else 0

decision = (
    "bc_contract_fit_ready_for_training_harness_wireup"
    if all_fit_gates and all_target_scope_ok and all_feature_coverage_ok
    else "bc_contract_readiness_failed_repair_before_training"
)
next_mechanism = (
    "wire_train_profile_to_fold_local_distillation_contract_then_run_full_fixed_wf_ytd_dashboard_replay"
    if decision == "bc_contract_fit_ready_for_training_harness_wireup"
    else "repair_bc_weighting_target_features_or_slice_thresholds_before_ppo"
)

output_paths = {
    "dataset_summary": CARD / "dataset_summary.csv",
    "fold_bc_metrics": CARD / "fold_bc_metrics.csv",
    "state_slice_bc_metrics": CARD / "state_slice_bc_metrics.csv",
    "bc_action_distribution": CARD / "bc_action_distribution.csv",
    "sample_weight_summary": CARD / "sample_weight_summary.csv",
    "feature_schema": CARD / "feature_schema.csv",
    "feature_summary": CARD / "feature_summary.json",
    "target_scope_audit": CARD / "target_scope_audit.csv",
    "selection": CARD / "selection.json",
    "summary": CARD / "summary.json",
}

selection = {
    "experiment": EXPERIMENT,
    "decision": decision,
    "next_mechanism": next_mechanism,
    "profile": profile.name,
    "source_contract": TARGET_CONTRACT,
    "source_decision": source_selection.get("decision"),
    "canonical_baseline": "soxlbacktest_policy_reimplementation_v1",
    "direct_comparison_baseline": "release16_checkpoint_distillation_ytd_alignment_contract_v1",
    "all_required_folds_evaluated": all_required_folds_evaluated,
    "all_fit_gates": all_fit_gates,
    "all_target_scope_ok": all_target_scope_ok,
    "all_feature_coverage_ok": all_feature_coverage_ok,
    "folds": fold_metrics.to_dict("records"),
}
write_json(CARD / "selection.json", selection)

retention_summary = retention_gate.get("summary", {}) if isinstance(retention_gate, dict) else {}
summary = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": decision,
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "role": "diagnose_bc_readiness",
    "canonical_baseline": "soxlbacktest_policy_reimplementation_v1",
    "direct_comparison_baseline": "release16_checkpoint_distillation_ytd_alignment_contract_v1",
    "target_window": list(WALK_FORWARD_WINDOWS),
    "changed_layer": ["model_selection", "training_objective_contract", "bc_warm_start_readiness"],
    "external_reference": "converted SOXLBacktest qlib buy/sell baseline plus release16 state-objective contract",
    "hypothesis": "Fold-local checkpoint distillation labels are useful only if the existing BC actor can fit the required state slices without label leakage before PPO.",
    "acceptance_criteria": [
        "target_scope_gate passes",
        "all required walk-forward folds have train-local labels",
        "feature frame covers all labels",
        "BC loss improves and required slices pass accuracy gates",
        "no actor retained or promoted from this screen",
    ],
    "expected_tradeoff": "Spend a short supervised readiness pass before launching a full retained-actor PPO run.",
    "failure_mode": "BC may fit ordinary labels while missing release/downtrend slices, causing another PPO run to underparticipate or drift above MaxDD limits.",
    "comparison_table_row_key": EXPERIMENT,
    "closest_historical_success_case": source_selection.get("closest_historical_success_case", "release16 wf_2022 BC initialization passed a walk-forward candidate gate"),
    "closest_historical_failure_case": source_selection.get("closest_historical_failure_case", "release16 state-head full actor failed YTD alignment and multiple WF gates"),
    "why_not_duplicate": "This screen does not rebuild the contract or rerun PPO; it verifies whether the latest fold-local target contract can be consumed by the existing BC training helper.",
    "profile": profile.name,
    "feature_summary": feature_summary,
    "target_scope_audit": scope_audit,
    "source_decision": source_selection.get("decision"),
    "retention_gate_current": {
        "passing_profile_count": retention_summary.get("passing_profile_count"),
        "reference_profile": retention_summary.get("reference_profile"),
        "reference_return_pct": retention_summary.get("reference_return_pct"),
        "current_best_return_pct": retention_summary.get("current_best_return_pct"),
        "required_best_return_pct": retention_summary.get("required_best_return_pct"),
        "max_walkforward_drawdown_pct": retention_summary.get("max_walkforward_drawdown_pct"),
    },
    "gate_summary": {
        "all_required_folds_evaluated": all_required_folds_evaluated,
        "all_fit_gates": all_fit_gates,
        "all_target_scope_ok": all_target_scope_ok,
        "all_feature_coverage_ok": all_feature_coverage_ok,
        "release_gate_pass_count": release_gate_pass_count,
        "required_fold_count": len(required_folds),
        "evaluated_fold_count": int(len(evaluated)),
    },
    "fold_bc_metrics": fold_metrics.to_dict("records"),
    "required_slice_rules": SLICE_RULES,
    "output_paths": {k: str(v.relative_to(ROOT)) for k, v in output_paths.items()},
    "next_mechanism": next_mechanism,
    "conclusion": (
        "Ready to wire the fold-local distillation contract into the training harness; still no actor promotion until fixed/WF/YTD/dashboard replay passes."
        if decision == "bc_contract_fit_ready_for_training_harness_wireup"
        else "Do not launch PPO yet; repair BC target weighting/features/slice coverage before training."
    ),
}
write_json(CARD / "summary.json", summary)

print(json.dumps(json_safe(summary["gate_summary"]), indent=2, sort_keys=True))
print("decision", decision)


{
  "all_feature_coverage_ok": true,
  "all_fit_gates": false,
  "all_required_folds_evaluated": true,
  "all_target_scope_ok": true,
  "evaluated_fold_count": 3,
  "release_gate_pass_count": 2,
  "required_fold_count": 3
}
decision bc_contract_readiness_failed_repair_before_training


## Conclusion

The next cell renders the machine-readable `summary.json` conclusion after execution. The notebook is the primary evidence for this single-run diagnostic; CSV/JSON files are sidecars for indexing and dashboard reuse.


In [6]:

summary = load_json(CARD / "summary.json")
lines = [
    f"Decision: `{summary['decision']}`.",
    f"Profile: `{summary['profile']}` with {summary['feature_summary']['feature_count']} features from packs `{summary['feature_summary']['feature_packs']}`.",
    f"Target scope gate: `{summary['gate_summary']['all_target_scope_ok']}`; feature coverage gate: `{summary['gate_summary']['all_feature_coverage_ok']}`; fit gate: `{summary['gate_summary']['all_fit_gates']}`.",
    f"Evaluated folds: {summary['gate_summary']['evaluated_fold_count']} / {summary['gate_summary']['required_fold_count']}; release-gate pass count: {summary['gate_summary']['release_gate_pass_count']}.",
    f"Next mechanism: `{summary['next_mechanism']}`.",
    summary["conclusion"],
]
display(Markdown("\n".join(f"- {line}" for line in lines)))


- Decision: `bc_contract_readiness_failed_repair_before_training`.
- Profile: `sparse_release_bc_ppo_screen_v1` with 59 features from packs `[]`.
- Target scope gate: `True`; feature coverage gate: `True`; fit gate: `False`.
- Evaluated folds: 3 / 3; release-gate pass count: 2.
- Next mechanism: `repair_bc_weighting_target_features_or_slice_thresholds_before_ppo`.
- Do not launch PPO yet; repair BC target weighting/features/slice coverage before training.